# Building Base Feature Table
This notebook is the backbone for understanding the data and assumptions made at the base feature building stage.

Purpose: The base feature table is ready for feature engineering (encoding, interactions, scaling, etc.) for modeling.

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

In [ ]:
%load_ext autoreload
%autoreload 2

from src.utils.load import load
from src.utils.data import tag_feature_map, add_suffix, valid_cols
from src.data.base_feature import make_constant, build_flag

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
df = load("data/canonical/events.parquet")
schema = load("configs/schema.yaml")
profile = load("configs/feature_profile.yaml")
tags = tag_feature_map(profile)

In [73]:
tags['minor_change']

['used_knowledge_base',
 'urgency_level',
 'impact_level',
 'priority_level',
 'category_id',
 'subcategory_id',
 'reported_symptom',
 'contact_channel']

## Inspect missing feature

In [74]:
missing_features = df.isna().sum()[df.isna().sum() > 0].index
missing_features

Index(['affected_uid', 'reported_by_uid', 'location_id', 'category_id',
       'subcategory_id', 'reported_symptom', 'asset_id', 'assigned_team_gid',
       'assigned_uid', 'root_cause_id', 'change_request_id', 'vendor_id',
       'caused_by_change_id', 'resolution_id', 'resolved_by_uid'],
      dtype='object')

In [75]:
# percentage of missing values by entries in data
missing_count_by_entries = df.isna().sum().sort_values(ascending=False)
missing_pct_by_entries = (100 * missing_count_by_entries / df.shape[0]).round(2).rename('%missing_by_entries')

# percentage of missing values by cases in data
missing_count_by_cases = df.isna().groupby(df['case_id']).any().sum().sort_values(ascending=False)
missing_pct_by_cases = (100 * missing_count_by_cases / df['case_id'].nunique()).round(2).rename("%missing_by_cases")

missing_pct_record = pd.concat([missing_pct_by_cases, missing_pct_by_entries], axis=1)
missing_pct_record.query("`%missing_by_cases` > 0 or `%missing_by_entries` > 0")

,%missing_by_cases,%missing_by_entries
caused_by_change_id,99.99,99.98
vendor_id,99.94,99.83
asset_id,99.80,99.69
change_request_id,99.61,99.30
root_cause_id,99.04,98.38
assigned_uid,35.15,19.40
reported_symptom,24.58,23.26
assigned_team_gid,15.47,10.03
reported_by_uid,1.01,0.97
resolution_id,0.43,0.50


In [76]:
missing_pct_record.query("`%missing_by_cases` > 99").index

Index(['caused_by_change_id', 'vendor_id', 'asset_id', 'change_request_id',
       'root_cause_id'],
      dtype='object')

The first 5 features are 99% nulls. 

* The null values in 'caused_by_change_id', 'change_request_id', 'root_cause_id' suggest the cases were not related to such relatd records
* Such cases did not caused by a change in IT sytem, and have no request for changing IT system so issue can be resolved, no IT problem is registered or known is related to the case.
--- from product and dataset information.

The 99% feels to sparse but they're genuine for a healthy business. 

Presence flag of such feature is capture for now to maintain the intent of feature.

## Handling (flag) 99% missing feature

In [77]:
tags = tag_feature_map(profile)
tags['sparse']

['caused_by_change_id',
 'change_request_id',
 'root_cause_id',
 'vendor_id',
 'asset_id']

In [78]:
add_suffix(tags['sparse'], "pflag", 2)

['caused_by_change_pflag',
 'change_request_pflag',
 'root_cause_pflag',
 'vendor_pflag',
 'asset_pflag']

In [79]:
sparse_cols = valid_cols(tags['sparse'], df)
df[add_suffix(sparse_cols, "pflag", 2)] = build_flag(df[sparse_cols], 
                                                          missing_flags=False)

In [80]:
# handled sparse value with flag 
df.drop(sparse_cols, axis=1, inplace=True)

## Inspect miniscule changing features

In [81]:
print(f"all {df.columns.size -1} features properties of 24_918 cases at case_level")
def generate_df_prop(df):
    grp = df.groupby('case_id')
    df_prop = pd.DataFrame(df.dtypes, columns=['dtype']).drop('case_id', axis=0)

    df_prop['no_of_unique'] = df.nunique(dropna=True)

    df_prop['no_of_changes'] = ((grp.nunique(dropna=True) <= 1).sum() - df.case_id.nunique(dropna=True)).round(2).abs()
    df_prop['pct_constant'] = (100* (grp.nunique(dropna=True) <= 1).sum() / df.case_id.nunique(dropna=True)).round(2).abs()

    df_prop['all_missing'] = df.isna().groupby(df['case_id']).all().sum()
    df_prop['any_missing'] = df.isna().groupby(df['case_id']).any().sum()
    return df_prop

df_prop = generate_df_prop(df)
df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes < 10')

all 35 features properties of 24_918 cases at case_level


,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
opened_at,datetime64[ns],19849,0,100.00,0,0
created_at,datetime64[ns],19559,0,100.00,0,0
notify_email,boolean,2,0,100.00,0,0
resolved_at,datetime64[ns],19500,0,100.00,0,0
closed_at,datetime64[ns],2707,0,100.00,0,0
created_at_is_imputed,bool,2,0,100.00,0,0
affected_uid,object,5244,0,100.00,3,3
location_id,object,224,0,100.00,6,6
resolved_by_uid,object,216,0,100.00,99,99
resolution_id,object,17,0,100.00,107,107


contact_channel:
- 5 out of 25k cases making constant (case_leve) feature changing (event_level).
- assumption: intial contact_channel is used for communication and later channel update.
- The later channels might be used for further communication like escalation, sharing info. Since they're only 5 cases, model won't generalize on them.

In [82]:
df_prop.sort_values(['any_missing', 'no_of_changes'], ascending=True).query('any_missing > 0')

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
affected_uid,object,5244,0,100.00,3,3
location_id,object,224,0,100.00,6,6
category_id,object,58,1184,95.25,7,7
subcategory_id,object,254,1738,93.03,8,8
resolved_by_uid,object,216,0,100.00,99,99
resolution_id,object,17,0,100.00,107,107
reported_by_uid,object,209,0,100.00,251,251
assigned_team_gid,object,78,9575,61.57,373,3855
reported_symptom,object,525,1323,94.69,5513,6126
assigned_uid,object,234,2858,88.53,658,8759


## Handling (drop) feature with minor missing cases


In [83]:
missing_table = df.isna().groupby(df['case_id']).all()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 0) & (missing_count < 10)].index
feature_names

Index(['affected_uid', 'location_id', 'category_id', 'subcategory_id'], dtype='object')

In [84]:
cases_to_drop = df.isna().groupby(df['case_id']).all()[feature_names].any(axis=1)
drop_ids = cases_to_drop[cases_to_drop].index

print(df[~df['case_id'].isin(drop_ids)].shape)

(141647, 36)


In [85]:
df = df[~df['case_id'].isin(drop_ids)]

In [86]:
print(df.shape, "original data shape")

(141647, 36) original data shape


## Handling (fill) featues missing major (1000s) cases

In [87]:
missing_table = df.isna().groupby(df['case_id']).any()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 1000)].index
feature_names

Index(['reported_symptom', 'assigned_team_gid', 'assigned_uid'], dtype='object')

In [88]:
(df[['reported_symptom', 'assigned_team_gid', 'assigned_uid']].isna()
 .groupby(df['case_id']).any().sum())

reported_symptom     6117
assigned_team_gid    3850
assigned_uid         8751
dtype: int64

In [89]:
(df[['reported_symptom', 'assigned_team_gid', 'assigned_uid']].isna()
 .groupby(df['case_id']).all().sum())

reported_symptom     5505
assigned_team_gid     368
assigned_uid          651
dtype: int64

In [90]:
df['reassignment_count'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27])

Checking assumption that the assigned agent changes with every reassignment count (or maybe along with reopen_count)

In [91]:
df.groupby(['case_id', 'reassignment_count'])['assigned_uid'].nunique(dropna=True).value_counts()

assigned_uid
1    38217
0     8361
2     1668
3       60
4        7
5        1
Name: count, dtype: int64

In [92]:
df.groupby(['case_id', 'reopen_count', 'reassignment_count'])['assigned_uid'].nunique(dropna=True).value_counts()

assigned_uid
1    38529
0     8366
2     1683
3       58
4        7
5        1
Name: count, dtype: int64

The number of uniques assigned agent more than 1 conclude that the previous assumption doesn't hold in the dataset..

In [93]:
# cardinality check
df[feature_names].nunique()

reported_symptom     525
assigned_team_gid     78
assigned_uid         233
dtype: int64

'reported_symptom','assigned_uid' are not highly missing values, and no valid product logic (reassignment/reopen count, features nmi) is found. 

- Forward or backward filling on assigned team or agent id will fabricate their pattern for model learning. This can be handled only by imputing "Unknown" value.
- reported_symptom is not a proper low cardinality (2% of the total cases), missing_flag likely to work best for this after forward fill. Assuming once the user has provided symptom/perception of issue it is known to the system/agent.

In [94]:
df[add_suffix('reported_symptom', "_mflag")] = build_flag(df['reported_symptom'], missing_flags=True) # capture original missingness
df['reported_symptom'] = df.groupby("case_id")['reported_symptom'].transform('ffill')
df['reported_symptom'] = df['reported_symptom'].fillna('Unknown')

reassignment count track the assigned team. refer to section ##[Inspect uid cols] for details and observation

In [95]:
grouped = df['assigned_team_gid'].groupby([df['case_id'], df['reassignment_count']])
df['assigned_team_gid'] = grouped.transform('ffill').fillna(grouped.transform('bfill'))

In [96]:
df['assigned_uid'] = df['assigned_uid'].fillna('Unknown')
df['assigned_team_gid'] = df['assigned_team_gid'].fillna('Unknown')

## Inspect features missing 100s of cases

In [97]:
missing_table = df.isna().groupby(df['case_id']).all()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 10) & (missing_count < 1000)].index.tolist()
feature_names

['reported_by_uid', 'resolution_id', 'resolved_by_uid']

resolution_id and resolved_by_uid columns are future values and will cause data leakage and are forbidden as model learning feature. 

In [98]:
df[feature_names].nunique()

reported_by_uid    208
resolution_id       17
resolved_by_uid    215
dtype: int64

## Handling (fill) 100s missing uid

In [99]:
tags['uid']

['affected_uid',
 'reported_by_uid',
 'resolved_by_uid',
 'updated_by_uid',
 'assigned_team_gid',
 'assigned_uid']

In [100]:
valid_cols(tags['uid'], df)

['affected_uid',
 'reported_by_uid',
 'resolved_by_uid',
 'updated_by_uid',
 'assigned_team_gid',
 'assigned_uid']

In [101]:
df_prop = generate_df_prop(df)
df_prop.loc[['affected_uid', 'reported_by_uid', 'resolved_by_uid', 'assigned_team_gid', 'assigned_uid']]

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
affected_uid,object,5244,0,100.00,0,0
reported_by_uid,object,208,0,100.00,250,250
resolved_by_uid,object,215,0,100.00,98,98
assigned_team_gid,object,79,11341,54.47,0,0
assigned_uid,object,234,8947,64.08,0,0


reported_by_uid and assigned_uid are low-medium cardinality (200s in 25k cases) and uid does have any ordering. 


Note: missing affected_uid, assigned uid, and assigned_team_gid are already handld and thus now 0.
- such cases with missing affected_uid are dropped [## Handling (drop) feature with minor missing cases]
- assigned uid and gid are filled with unknown. [## Handling (fill) featues missing major (1000s) cases]

Resolved by is final outcome. not input/learning feature. It won't be touched.

In [102]:
df['reported_by_uid'] = df['reported_by_uid'].fillna("Unknown")

## Handling features - minor changing within cases.

In [103]:
df_prop = generate_df_prop(df)
df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes > 0 and pct_constant > 50')

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
caused_by_change_pflag,int64,2,1,100.00,0,0
contact_channel,object,5,5,99.98,0,0
asset_pflag,int64,2,5,99.98,0,0
vendor_pflag,int64,2,52,99.79,0,0
change_request_pflag,int64,2,82,99.67,0,0
root_cause_pflag,int64,2,143,99.43,0,0
used_knowledge_base,bool,2,208,99.16,0,0
reopen_count,int64,9,275,98.90,0,0
urgency_level,object,3,299,98.80,0,0
impact_level,object,3,315,98.74,0,0


Flags are meant to capture the intent from original data, they won't be made case_level constant

Following features are likely genuine change and real signal:
- used_knowledge_base: ex, first knowledge not used, later accessed for further investigation.
- reopen_count: only 1% cases are reopened. 
- priority_level: ex, user raised the urgency later to reduce deadline; agent changed impact upon case investigation. 
- reopen_count is likely weak feature for model.

similar senarios can be listed for others in the list.

## Inspect transition of features 10% changing (>90% constant).

In [104]:
def summarize_transitions(df, col):
    nunique = df.groupby('case_id')[col].nunique()
    varying_ids = nunique[nunique > 1].index  # pre-filter
    
    return (
        df[df['case_id'].isin(varying_ids)]
        .sort_values(['case_id', 'updated_at', 'system_update_count'])
        .groupby('case_id')[col]
        .apply(lambda x: f"{x.iloc[0]} → {x.iloc[-2]}") # last event is mostly Closed status
        .value_counts()
    )

In [105]:
summarize_transitions(df, 'used_knowledge_base')

used_knowledge_base
True → False    106
False → True    102
Name: count, dtype: int64

Both true to false, and false to true have almost same distribution. 

Because they are changing in only 1% of cases. For this, case_level constant proxy to be created using first value from the cases.

In [106]:
summarize_transitions(df, 'urgency_level')

urgency_level
2 - Medium → 1 - High      195
1 - High → 2 - Medium       48
2 - Medium → 3 - Low        18
1 - High → 3 - Low          14
2 - Medium → 2 - Medium     13
3 - Low → 2 - Medium         6
3 - Low → 1 - High           3
1 - High → 1 - High          2
Name: count, dtype: int64

In [107]:
summarize_transitions(df, 'impact_level')

impact_level
2 - Medium → 1 - High      194
1 - High → 2 - Medium       51
2 - Medium → 3 - Low        24
2 - Medium → 2 - Medium     21
1 - High → 3 - Low          12
3 - Low → 2 - Medium         7
3 - Low → 1 - High           4
1 - High → 1 - High          2
Name: count, dtype: int64

In [108]:
summarize_transitions(df, 'priority_level')

priority_level
3 - Moderate → 2 - High        123
3 - Moderate → 1 - Critical    123
1 - Critical → 3 - Moderate     35
3 - Moderate → 4 - Low          22
1 - Critical → 4 - Low          18
2 - High → 1 - Critical         16
3 - Moderate → 3 - Moderate     13
2 - High → 3 - Moderate         11
4 - Low → 3 - Moderate           7
1 - Critical → 2 - High          5
4 - Low → 1 - Critical           4
2 - High → 4 - Low               4
1 - Critical → 1 - Critical      2
Name: count, dtype: int64

The adjustment in priority (urgency/impact) level was performed majorly to escalate the case.


## Handle <10% changing features - constant proxy

convert minor changing features so model can learn that these are constant case_level feature. 

first event data will be used to convert feature to constant case_level.

Modeling will be experimented with both - original and constant-proxy features.

In [109]:
df_prop.query('100 > pct_constant > 90').sort_values('pct_constant')

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
subcategory_id,object,254,1737,93.03,0,0
reported_symptom,object,526,1531,93.85,0,0
category_id,object,57,1182,95.25,0,0
reported_symptom_mflag,int64,2,612,97.54,0,0
priority_level,object,4,383,98.46,0,0
impact_level,object,3,315,98.74,0,0
urgency_level,object,3,299,98.80,0,0
reopen_count,int64,9,275,98.90,0,0
used_knowledge_base,bool,2,208,99.16,0,0
root_cause_pflag,int64,2,143,99.43,0,0


In [110]:
# removed flag as it capture original data
minor_changing_feature = valid_cols(tags['minor_change'], df)
minor_changing_feature

cproxy_feature = add_suffix(minor_changing_feature, "_cproxy") #constant proxy features
cproxy_feature

['used_knowledge_base_cproxy',
 'urgency_level_cproxy',
 'impact_level_cproxy',
 'priority_level_cproxy',
 'category_id_cproxy',
 'subcategory_id_cproxy',
 'reported_symptom_cproxy',
 'contact_channel_cproxy']

In [111]:
df[cproxy_feature] = make_constant(df, minor_changing_feature)
df.shape

(141647, 45)

In [112]:
(df[cproxy_feature].groupby(df['case_id']).nunique() > 1).sum()

used_knowledge_base_cproxy    0
urgency_level_cproxy          0
impact_level_cproxy           0
priority_level_cproxy         0
category_id_cproxy            0
subcategory_id_cproxy         0
reported_symptom_cproxy       0
contact_channel_cproxy        0
dtype: int64

## Ordinal Feature Encoding

In [114]:
from src.data.base_features import build_base_features
build_base_features(return_events=True).shape

(141647, 45)

In [115]:
profile['columns']['urgency_level']['mapping']

{'3 - Low': 1, '2 - Medium': 2, '1 - High': 3}

In [116]:
valid_cols(tags['ordinal'], df)

['urgency_level', 'impact_level', 'priority_level']

In [117]:
df['impact_encoded'] = df['impact_level'].map({'3 - Low': 1, '2 - Medium': 2, '1 - High': 3})
df.drop('impact_level', axis=1, inplace=True)
df['urgency_encoded'] = df['urgency_level'].map({'3 - Low': 1, '2 - Medium': 2, '1 - High': 3})
df.drop('urgency_level', axis=1, inplace=True)
df["priority_encoded"] = df["priority_level"].map({"1 - Critical":  4,"2 - High": 3, "3 - Moderate": 2, "4 - Low": 1})
df.drop("priority_level", axis=1, inplace=True)

## boolean features to int

In [118]:
df.select_dtypes(include=bool)

,is_open,met_deadline,used_knowledge_base,priority_confirmed,notify_email,created_at_is_imputed,used_knowledge_base_cproxy
0,True,True,True,False,False,False,True
1,True,True,True,False,False,False,True
2,True,True,True,False,False,False,True
3,False,True,True,False,False,False,True
4,True,True,True,False,False,False,True
...,...,...,...,...,...,...,...
141707,False,True,False,True,False,True,False
141708,True,True,False,False,False,True,False
141709,True,True,False,False,False,True,False
141710,True,True,False,True,False,True,False


In [119]:
booleans = valid_cols(tags['boolean'], df)
booleans.remove('met_deadline')

In [126]:
bool_df = df.select_dtypes(bool).copy() # for original and dervied features
bool_df.drop(['met_deadline'], inplace=True, axis=1)
df[bool_df.columns] = bool_df.astype(int)

## Assess notebook and modular code (py) result

In [128]:
from src.data.base_features import build_base_features
df_from_py = build_base_features(return_events=True)
df_from_py.shape

(141647, 45)

In [129]:
print("shape:", df.shape, df_from_py.shape, "\n",
      "different index:", list(set(df.index) ^ set(df_from_py.index)), "\n",
      "different columns:", list(set(df.columns) ^ set(df_from_py.columns))
      )

shape: (141647, 45) (141647, 45) 
 different index: [] 
 different columns: []


In [130]:
df1 = df.sort_index().sort_index(axis=1)
df2 = df_from_py.sort_index().sort_index(axis=1)

df1.compare(df2)
df1.equals(df2)

True

The imputation performed in notebook and modular code are producing same result as indended.

## Inspect target-driving/outcome

In [121]:
df = load("data\canonical\events.parquet")
df.shape

(141712, 38)

In [122]:
outcome_df = df[["case_id"] +tags['guardrail']]

In [123]:
outcome_df.shape

(141712, 6)

In [124]:
generate_df_prop(outcome_df)

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
resolved_at,datetime64[ns],19500,0,100.00,0,0
closed_at,datetime64[ns],2707,0,100.00,0,0
resolved_by_uid,object,216,0,100.00,99,99
resolution_id,object,17,0,100.00,107,107
met_deadline,bool,2,9114,63.42,0,0


In [125]:
outcome_df[outcome_df['resolution_id'].isna()]

,case_id,resolved_at,closed_at,resolved_by_uid,resolution_id,met_deadline
1520,INC0000519,2016-03-02 11:07:00,2016-03-07 12:00:00,Resolved by 215,None,True
1521,INC0000519,2016-03-02 11:07:00,2016-03-07 12:00:00,Resolved by 215,None,True
1522,INC0000519,2016-03-02 11:07:00,2016-03-07 12:00:00,Resolved by 215,None,True
1523,INC0000519,2016-03-02 11:07:00,2016-03-07 12:00:00,Resolved by 215,None,True
1524,INC0000519,2016-03-02 11:07:00,2016-03-07 12:00:00,Resolved by 215,None,True
...,...,...,...,...,...,...
141699,INC0120495,2017-02-16 09:51:00,2017-02-16 09:51:00,None,None,True
141700,INC0120495,2017-02-16 09:51:00,2017-02-16 09:51:00,None,None,True
141701,INC0120495,2017-02-16 09:51:00,2017-02-16 09:51:00,None,None,True
141702,INC0120495,2017-02-16 09:51:00,2017-02-16 09:51:00,None,None,True


In [127]:
df['case_status'].unique()

array(['New', 'Resolved', 'Closed', 'Active', 'Awaiting User Info',
       'Awaiting Problem', 'Awaiting Vendor', 'Awaiting Evidence',
       'Unknown'], dtype=object)

the resolution code (id) and resolved_by_uid are final outcome of a case.

However, they do not add any value to decision making or business prediction.

## Inspect uid cols 

In [4]:
df = load('data/canonical/events.parquet')

In [132]:
from src.data.base_features import build_base_features
df_processed = build_base_features(return_events=True)
df_processed = df_processed.replace('Unknown' or "unknown", np.nan)
df_processed[tags['uid']].isna().sum()

affected_uid             0
reported_by_uid       1365
resolved_by_uid        223
updated_by_uid           0
assigned_team_gid    14192
assigned_uid         27467
dtype: int64

In [6]:
df[tags['uid']].isna().sum()

affected_uid            29
reported_by_uid       1378
resolved_by_uid        226
updated_by_uid           0
assigned_team_gid     4187
assigned_uid         27496
dtype: int64

The 29 missing affected uid were coming rows which we already handled (dropped - only 3 cases).

In [9]:
generate_df_prop(df_processed[['case_id'] + tags['uid']])

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
affected_uid,object,5244,0,100.00,0,0
reported_by_uid,object,208,0,100.00,250,250
resolved_by_uid,object,215,0,100.00,98,98
updated_by_uid,object,846,23369,6.17,0,0
assigned_team_gid,object,78,9703,61.04,0,2534
assigned_uid,object,233,2856,88.53,651,8751


assigned group id , and assigned user id have higher missing values.

### assessing assigned_uid and assigned_team_gid

From dataset description, the reopen count and reassignment tracks cases and team/user status. invesitgating further to find any relationship to fill and impute missing values

In [17]:
cases = df.query('reopen_count > 0')['case_id'].unique()
reopened_cases = df[df['case_id'].isin(cases)] # index of cases with reopen > 0
(reopened_cases['case_status'] == 'Closed').groupby(reopened_cases['case_id']).sum().value_counts()

case_status
1    261
2     12
3      2
Name: count, dtype: int64

In [15]:
(df['case_status'] == 'Closed').groupby(df['case_id']).sum().value_counts()

case_status
1    24861
2       48
3        8
4        1
Name: count, dtype: int64

cases can have multiple entries/events with closed status even if it is never reopened.

In [72]:
df['reopen_count'].max()

np.int64(8)

In [130]:
df.query('7 <= reopen_count')['case_id'].unique()

array(['INC0019396'], dtype=object)

In [131]:
df.query('case_id == "INC0019396"')[['case_id', 'case_status', "reopen_count", 'reassignment_count'] + tags['uid']]

,case_id,case_status,reopen_count,reassignment_count,affected_uid,reported_by_uid,resolved_by_uid,updated_by_uid,assigned_team_gid,assigned_uid
81418,INC0019396,New,0,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 908,Group 66,None
81419,INC0019396,New,0,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 278,Group 66,None
81420,INC0019396,Resolved,0,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 713,Group 66,Resolver 175
81421,INC0019396,Active,1,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 278,Group 66,Resolver 175
81422,INC0019396,Active,1,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 278,Group 66,Resolver 175
81423,INC0019396,Active,1,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 278,Group 66,Resolver 175
81424,INC0019396,Active,1,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 278,Group 66,Resolver 175
81425,INC0019396,Active,1,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 278,Group 66,Resolver 175
81426,INC0019396,Active,1,0,Caller 1580,Opened by 305,Resolved by 158,Updated by 278,Group 66,Resolver 175
81427,INC0019396,Active,1,1,Caller 1580,Opened by 305,Resolved by 158,Updated by 713,Group 54,None


Based on the flow of event in cases:

reopen count track when a case switch from resolved/closed status to active.

reassignment count track when a case has transition or change in assignment group.

### Imputing assigned_team_gid using relationship with reassignment_count

In [22]:
df['assigned_team_gid'].isna().groupby([df['case_id'], df['reassignment_count']]).agg(['any', 'all']).sum()

any    2740
all    1122
dtype: int64

In [28]:
len(df.groupby(['case_id', 'reassignment_count']))

48332

In [29]:
grouped = df['assigned_team_gid'].groupby([df['case_id'], df['reassignment_count']])
temp = grouped.transform('ffill').fillna(grouped.transform('bfill'))

In [31]:
print("missing data in original df:", df['assigned_team_gid'].isna().sum(), "\n"
      " missing data after ffill + bfill:", temp.isna().sum())

missing data in original df: 4187 
 missing data after ffill + bfill: 1525


The reduction in null through reassignment relation is stasticially significant. 

However, we need to check the assumption whether group changes within the reassignment_count for any case.

In [39]:
df['assigned_team_gid'].groupby([df['case_id'], df['reassignment_count']]).nunique(dropna=True).value_counts()

assigned_team_gid
1    47172
0     1122
2       36
3        2
Name: count, dtype: int64

In [51]:
df['assigned_team_gid'].groupby([df['case_id'], df['reassignment_count']]).ngroups

48332

While, 38 segments are invalidating the assumuption - logging or importing inconsistency, or malfunction of counter could be a cause of those anomaly. Also, they are only 38 out of 48332 which will not materially affect modeling.

In [61]:
df[df[['resolution_id']].isna().values][['case_id'] + tags['case_state'] + ['resolution_id']].head(10)

,case_id,case_status,is_open,reassignment_count,reopen_count,met_deadline,resolution_id
1520,INC0000519,New,1,0,0,True,None
1521,INC0000519,New,1,0,0,True,None
1522,INC0000519,Active,1,0,0,True,None
1523,INC0000519,Active,1,0,0,True,None
1524,INC0000519,Resolved,1,0,0,True,None
1525,INC0000519,Closed,0,0,0,True,None
3044,INC0000839,New,1,0,0,True,None
3045,INC0000839,Active,1,0,0,True,None
3046,INC0000839,Active,1,0,0,True,None
3047,INC0000839,Resolved,1,0,0,True,None


In [63]:
df.head(4)[['case_id'] + tags['case_state'] + ['resolution_id']]

,case_id,case_status,is_open,reassignment_count,reopen_count,met_deadline,resolution_id
0,INC0000045,New,1,0,0,True,code 5
1,INC0000045,Resolved,1,0,0,True,code 5
2,INC0000045,Resolved,1,0,0,True,code 5
3,INC0000045,Closed,0,0,0,True,code 5


In [66]:
df.dtypes.value_counts()

object            21
int64             17
datetime64[ns]     5
bool               2
float64            2
Name: count, dtype: int64

In [68]:
df.select_dtypes(include=object).columns

Index(['case_id', 'case_status', 'affected_uid', 'reported_by_uid',
       'updated_by_uid', 'contact_channel', 'location_id', 'category_id',
       'subcategory_id', 'reported_symptom', 'assigned_team_gid',
       'assigned_uid', 'resolution_id', 'resolved_by_uid',
       'urgency_level_cproxy', 'impact_level_cproxy', 'priority_level_cproxy',
       'category_id_cproxy', 'subcategory_id_cproxy',
       'reported_symptom_cproxy', 'contact_channel_cproxy'],
      dtype='object')